### N-gram language models or how to write scientific papers (4 pts)

We shall train our language model on a corpora of [ArXiv](http://arxiv.org/) articles and see if we can generate a new one!

![img](https://media.npr.org/assets/img/2013/12/10/istock-18586699-monkey-computer_brick-16e5064d3378a14e0e4c2da08857efe03c04695e-s800-c85.jpg)

_data by neelshah18 from [here](https://www.kaggle.com/neelshah18/arxivdataset/)_

_Disclaimer: this has nothing to do with actual science. But it's fun, so who cares?!_

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

In [5]:
data = pd.read_json("./arxivData.json")
data.sample(n=5)

,author,day,id,link,month,summary,tag,title,year
19994,"[{'name': 'Kerstin Schill'}, {'name': 'Ernst P...",20,1303.5748v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",3,A control strategy for expert systems is prese...,"[{'term': 'cs.AI', 'scheme': 'http://arxiv.org...",Completing Knowledge by Competing Hierarchies,2013
22895,"[{'name': 'Pascal Lu'}, {'name': 'Olivier Coll...",10,1710.03627v1,"[{'rel': 'related', 'href': 'http://dx.doi.org...",10,"In this paper, we propose a framework for auto...","[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Multilevel Modeling with Structured Penalties ...,2017
8273,"[{'name': 'Fang Zhao'}, {'name': 'Yongzhen Hua...",26,1501.06272v2,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",1,"With the rapid growth of web images, hashing h...","[{'term': 'cs.CV', 'scheme': 'http://arxiv.org...",Deep Semantic Ranking Based Hashing for Multi-...,2015
22537,"[{'name': 'Aleksander Wieczorek'}, {'name': 'V...",1,1611.00261v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",11,We propose a new method of discovering causal ...,"[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Causal Compression,2016
7456,"[{'name': 'Mu Niu'}, {'name': 'Pokman Cheung'}...",3,1801.01061v1,"[{'rel': 'alternate', 'href': 'http://arxiv.or...",1,We propose a class of intrinsic Gaussian proce...,"[{'term': 'stat.ML', 'scheme': 'http://arxiv.o...",Intrinsic Gaussian processes on complex constr...,2018


In [6]:
# assemble lines: concatenate title and description
lines = data.apply(lambda row: row['title'] + ' ; ' + row['summary'].replace("\n", ' '), axis=1).tolist()

sorted(lines, key=len)[:3]

['Differential Contrastive Divergence ; This paper has been retracted.',
 'What Does Artificial Life Tell Us About Death? ; Short philosophical essay',
 'P=NP ; We claim to resolve the P=?NP problem via a formal argument for P=NP.']

### Tokenization

You know the dril. The data is messy. Go clean the data. Use WordPunctTokenizer or something.


In [7]:
# Task: convert lines (in-place) into strings of space-separated tokens. Import & use WordPunctTokenizer
from nltk import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

lines = [' '.join(tokenizer.tokenize(line.lower())) for line in lines]

print(lines[3])

learning what to share between loosely related tasks ; multi - task learning is motivated by the observation that humans bring to bear what they know about related problems when solving new ones . similarly , deep neural networks can profit from related tasks by sharing parameters with other networks . however , humans do not consciously decide to transfer knowledge between tasks . in natural language processing ( nlp ), it is hard to predict if sharing will lead to improvements , particularly if tasks are only loosely related . to overcome this , we introduce sluice networks , a general framework for multi - task learning where trainable parameters control the amount of sharing . our framework generalizes previous proposals in enabling sharing of all combinations of subspaces , layers , and skip connections . we perform experiments on three task pairs , and across seven different domains , using data from ontonotes 5 . 0 , and achieve up to 15 % average error reductions over common ap

In [8]:
assert sorted(lines, key=len)[0] == \
    'differential contrastive divergence ; this paper has been retracted .'
assert sorted(lines, key=len)[2] == \
    'p = np ; we claim to resolve the p =? np problem via a formal argument for p = np .'

### N-Gram Language Model (1point)

A language model is a probabilistic model that estimates text probability: the joint probability of all tokens $w_t$ in text $X$: $P(X) = P(w_1, \dots, w_T)$.

It can do so by following the chain rule:
$$P(w_1, \dots, w_T) = P(w_1)P(w_2 \mid w_1)\dots P(w_T \mid w_1, \dots, w_{T-1}).$$ 

The problem with such approach is that the final term $P(w_T \mid w_1, \dots, w_{T-1})$ depends on $n-1$ previous words. This probability is impractical to estimate for long texts, e.g. $T = 1000$.

One popular approximation is to assume that next word only depends on a finite amount of previous words:

$$P(w_t \mid w_1, \dots, w_{t - 1}) = P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1})$$

Such model is called __n-gram language model__ where n is a parameter. For example, in 3-gram language model, each word only depends on 2 previous words. 

$$
    P(w_1, \dots, w_n) = \prod_t P(w_t \mid w_{t - n + 1}, \dots, w_{t - 1}).
$$

You can also sometimes see such approximation under the name of _n-th order markov assumption_.

The first stage to building such a model is counting all word occurences given N-1 previous words

In [9]:
from tqdm import tqdm
from collections import defaultdict, Counter

# special tokens: 
# - `UNK` represents absent tokens, 
# - `EOS` is a special token after the end of sequence

UNK, EOS = "_UNK_", "_EOS_"

def count_ngrams(lines, n):
    """
    Count how many times each word occured after (n - 1) previous words
    :param lines: an iterable of strings with space-separated tokens
    :returns: a dictionary { tuple(prefix_tokens): {next_token_1: count_1, next_token_2: count_2}}

    When building counts, please consider the following two edge cases:
    - if prefix is shorter than (n - 1) tokens, it should be padded with UNK. For n=3,
      empty prefix: "" -> (UNK, UNK)
      short prefix: "the" -> (UNK, the)
      long prefix: "the new approach" -> (new, approach)
    - you should add a special token, EOS, at the end of each sequence
      "... with deep neural networks ." -> (..., with, deep, neural, networks, ., EOS)
      count the probability of this token just like all others.
    """
    counts = defaultdict(Counter)
    # counts[(word1, word2)][word3] = how many times word3 occured after (word1, word2)

    for line in lines:
      line = line.split()
      for _ in range(n-1):
        line.insert(0, UNK)
      line.append(EOS)
      # print(line)
      for i in range(len(line) - n + 1):
        prefix = tuple(line[index+i] for index in range(n-1))
        next_token = line[n + i - 1]
        
        counts[prefix][next_token] +=1
    
    return counts


In [10]:
# let's test it
dummy_lines = sorted(lines, key=len)[:100]
dummy_counts = count_ngrams(dummy_lines, n=3)
assert set(map(len, dummy_counts.keys())) == {2}, "please only count {n-1}-grams"
assert len(dummy_counts[('_UNK_', '_UNK_')]) == 78
assert dummy_counts['_UNK_', 'a']['note'] == 3
assert dummy_counts['p', '=']['np'] == 2
assert dummy_counts['author', '.']['_EOS_'] == 1

Once we can count N-grams, we can build a probabilistic language model.
The simplest way to compute probabilities is in proporiton to counts:

$$ P(w_t | prefix) = { Count(prefix, w_t) \over \sum_{\hat w} Count(prefix, \hat w) } $$

In [11]:
class NGramLanguageModel:    
    def __init__(self, lines, n):
        """ 
        Train a simple count-based language model: 
        compute probabilities P(w_t | prefix) given ngram counts
        
        :param n: computes probability of next token given (n - 1) previous words
        :param lines: an iterable of strings with space-separated tokens
        """
        assert n >= 1
        self.n = n
    
        counts = count_ngrams(lines, self.n)
        
        # compute token proabilities given counts
        self.probs = defaultdict(Counter)
        # probs[(word1, word2)][word3] = P(word3 | word1, word2)
        
        # populate self.probs with actual probabilities
        for prefix, count_dict in counts.items():
            total_count = sum(count_dict.values())
                
            for word, word_count in count_dict.items():
                self.probs[prefix][word] = word_count / total_count
            
    def get_possible_next_tokens(self, prefix):
        """
        :param prefix: string with space-separated prefix tokens
        :returns: a dictionary {token : it's probability} for all tokens with positive probabilities
        """
        prefix = prefix.split()
        prefix = prefix[max(0, len(prefix) - self.n + 1):]
        prefix = [ UNK ] * (self.n - 1 - len(prefix)) + prefix
        return self.probs[tuple(prefix)]
    
    def get_next_token_prob(self, prefix, next_token):
        """
        :param prefix: string with space-separated prefix tokens
        :param next_token: the next token to predict probability for
        :returns: P(next_token|prefix) a single number, 0 <= P <= 1
        """
        return self.get_possible_next_tokens(prefix).get(next_token, 0)

In [13]:
dummy_lm = NGramLanguageModel(lines, n=4)

prefix = 'a language model'

print(end=prefix + ' ', flush=True)
next_token = ''
while next_token != EOS:
    tokens, probs = zip(*dummy_lm.get_possible_next_tokens(prefix).items())
    next_token = np.random.choice(tokens, p=probs)
    prefix = prefix + ' ' + next_token
    print(end=next_token + ' ', flush=True)

a language model and reduces the training time . second , xnv applies multiview regression using canonical correlation analysis . we obtain encouraging results from evaluating their performance by using k - means clustering algorithm is applied successfully to recurrent models , capturing these biases requires the use of expensive iterative methods to unique fixed points , the presence of multiple agents and the expertism helps the performance and applicability strongly depends on the dimensions of input feature learning and dictionary - blind image restoration problems , including multilabel adaboost , convex multitask feature learning , others pretrain the policies by imitating expert demonstrations ) while simultaneously ensuring privacy of the original problem in any way , and examine the errors associated with detection of changes and recovery of structures . we also analyze the effect of using variants of ltl to specify the region of interest using a low - dimensional nonlinear m

Let's test it!

In [14]:
dummy_lm = NGramLanguageModel(dummy_lines, n=3)

p_initial = dummy_lm.get_possible_next_tokens('') # '' -> ['_UNK_', '_UNK_']
assert np.allclose(p_initial['learning'], 0.02)
assert np.allclose(p_initial['a'], 0.13)
assert np.allclose(p_initial.get('meow', 0), 0)
assert np.allclose(sum(p_initial.values()), 1)

p_a = dummy_lm.get_possible_next_tokens('a') # '' -> ['_UNK_', 'a']
assert np.allclose(p_a['machine'], 0.15384615)
assert np.allclose(p_a['note'], 0.23076923)
assert np.allclose(p_a.get('the', 0), 0)
assert np.allclose(sum(p_a.values()), 1)

assert np.allclose(dummy_lm.get_possible_next_tokens('a note')['on'], 1)
assert dummy_lm.get_possible_next_tokens('a machine') == \
    dummy_lm.get_possible_next_tokens("there have always been ghosts in a machine"), \
    "your 3-gram model should only depend on 2 previous words"

Now that you've got a working n-gram language model, let's see what sequences it can generate. But first, let's train it on the whole dataset.

In [15]:
lm = NGramLanguageModel(lines, n=3)

The process of generating sequences is... well, it's sequential. You maintain a list of tokens and iteratively add next token by sampling with probabilities.

$ X = [] $

__forever:__
* $w_{next} \sim P(w_{next} | X)$
* $X = concat(X, w_{next})$


Instead of sampling with probabilities, one can also try always taking most likely token, sampling among top-K most likely tokens or sampling with temperature. In the latter case (temperature), one samples from

$$w_{next} \sim {P(w_{next} | X) ^ {1 / \tau} \over \sum_{\hat w} P(\hat w | X) ^ {1 / \tau}}$$

Where $\tau > 0$ is model temperature. If $\tau << 1$, more likely tokens will be sampled with even higher probability while less likely tokens will vanish.

In [16]:
from math import pow

def get_next_token(lm, prefix, temperature=1.0):
    """
    return next token after prefix;
    :param temperature: samples proportionally to lm probabilities ^ (1 / temperature)
        if temperature == 0, always takes most likely token. Break ties arbitrarily.
    """

    possible_token = lm.get_possible_next_tokens(prefix)
    
    if temperature == 0:
        return max(possible_token, key=possible_token.get)
    
    scaled_probs = {word: pow(proba, 1/temperature) for word, proba in possible_token.items()}
        
    proba_sum = sum(scaled_probs.values())
    tokens = list(scaled_probs.keys())
    probabilities = [p / proba_sum for p in scaled_probs.values()]
        
    return np.random.choice(tokens, p=probabilities)
    
    

In [17]:
from collections import Counter
test_freqs = Counter([get_next_token(lm, 'there have') for _ in range(10000)])
assert 250 < test_freqs['not'] < 450
assert 8500 < test_freqs['been'] < 9500
assert 1 < test_freqs['lately'] < 200

test_freqs = Counter([get_next_token(lm, 'deep', temperature=1.0) for _ in range(10000)])
assert 1500 < test_freqs['learning'] < 3000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.5) for _ in range(10000)])
assert 8000 < test_freqs['learning'] < 9000
test_freqs = Counter([get_next_token(lm, 'deep', temperature=0.0) for _ in range(10000)])
assert test_freqs['learning'] == 10000

print("Looks nice!")

Looks nice!


Let's have fun with this model

In [18]:
prefix = 'artificial intelligence' # <- your ideas :)

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

artificial intelligence ( ai ). the new measure for dynamic sampling at prediction time scale . the resulting representations are often spatially varying . the algebra defines labeled disjunctive metric constraints in a cooperative intention task . we compare the following three principal contributions . the best of our method ( second - order statistics based on anchor points are preserved at a cost - sensitive embeddings in a joint 3dcnn - dqn ( ldqn ) empirically against the subset selection , where one seeks to reconstruct and estimate their centers and the distances themselves . the system fails to be chosen


In [19]:
prefix = 'bridging the gap' # <- more of your ideas

for i in range(100):
    prefix += ' ' + get_next_token(lm, prefix, temperature=0.5)
    if prefix.endswith(EOS) or len(lm.get_possible_next_tokens(prefix)) == 0:
        break
        
print(prefix)

bridging the gap between synthetic and real - time by more than two independent latent variables in the form of the state - of - the - art methods . _EOS_


__More in the homework:__ nucleus sampling, top-k sampling, beam search(not for the faint of heart).

### Evaluating language models: perplexity (1point)

Perplexity is a measure of how well your model approximates the true probability distribution behind the data. __Smaller perplexity = better model__.

To compute perplexity on one sentence, use:
$$
    {\mathbb{P}}(w_1 \dots w_N) = P(w_1, \dots, w_N)^{-\frac1N} = \left( \prod_t P(w_t \mid w_{t - n}, \dots, w_{t - 1})\right)^{-\frac1N},
$$


On the corpora level, perplexity is a product of probabilities of all tokens in all sentences to the power of $1/N$, where $N$ is __total length (in tokens) of all sentences__ in corpora.

This number can quickly get too small for float32/float64 precision, so we recommend you to first compute log-perplexity (from log-probabilities) and then take the exponent.

In [20]:
def perplexity(lm, lines, min_logprob=np.log(10 ** -50.)):
    """
    :param lines: a list of strings with space-separated tokens
    :param min_logprob: if log(P(w | ...)) is smaller than min_logprop, set it equal to min_logrob
    :returns: corpora-level perplexity - a single scalar number from the formula above
    
    Note: do not forget to compute P(w_first | empty) and P(eos | full_sequence)
    
    PLEASE USE lm.get_next_token_prob and NOT lm.get_possible_next_tokens
    """
    log_perplexity = 0
    token_count = 0
    for line in lines:
        words = line.split()
        words.append(EOS)
        token_count += len(words)
        for i in range(len(words)):
            before_sentence = ' '.join(words[:i])
            proba = lm.get_next_token_prob(before_sentence, words[i])
            log_proba = max(np.log(proba), min_logprob)
            log_perplexity += log_proba
            
    perplexity = np.exp(-log_perplexity/token_count)
    
    
    return perplexity

In [21]:
lm1 = NGramLanguageModel(dummy_lines, n=1)
lm3 = NGramLanguageModel(dummy_lines, n=3)
lm10 = NGramLanguageModel(dummy_lines, n=10)

ppx1 = perplexity(lm1, dummy_lines)
ppx3 = perplexity(lm3, dummy_lines)
ppx10 = perplexity(lm10, dummy_lines)
ppx_missing = perplexity(lm3, ['the jabberwock , with eyes of flame , '])  # thanks, L. Carrol

print("Perplexities: ppx1=%.3f ppx3=%.3f ppx10=%.3f" % (ppx1, ppx3, ppx10))

assert all(0 < ppx < 500 for ppx in (ppx1, ppx3, ppx10)), "perplexity should be non-negative and reasonably small"
assert ppx1 > ppx3 > ppx10, "higher N models should overfit and "
assert np.isfinite(ppx_missing) and ppx_missing > 10 ** 6, "missing words should have large but finite perplexity. " \
    " Make sure you use min_logprob right"
assert np.allclose([ppx1, ppx3, ppx10], (318.2132342216302, 1.5199996213739575, 1.1838145037901249))

Perplexities: ppx1=318.213 ppx3=1.520 ppx10=1.184


C:\Users\user\AppData\Local\Temp\ipykernel_22028\1798100124.py:20: RuntimeWarning: divide by zero encountered in log
  log_proba = max(np.log(proba), min_logprob)


Now let's measure the actual perplexity: we'll split the data into train and test and score model on test data only.

In [22]:
from sklearn.model_selection import train_test_split
train_lines, test_lines = train_test_split(lines, test_size=0.25, random_state=42)

for n in (1, 2, 3):
    lm = NGramLanguageModel(n=n, lines=train_lines)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))


C:\Users\user\AppData\Local\Temp\ipykernel_22028\1798100124.py:20: RuntimeWarning: divide by zero encountered in log
  log_proba = max(np.log(proba), min_logprob)


N = 1, Perplexity = 1832.23136
N = 2, Perplexity = 85653987.28774
N = 3, Perplexity = 61999196259043346743296.00000


In [ ]:
# whoops, it just blew up :)

### LM Smoothing

The problem with our simple language model is that whenever it encounters an n-gram it has never seen before, it assigns it with the probabilitiy of 0. Every time this happens, perplexity explodes.

To battle this issue, there's a technique called __smoothing__. The core idea is to modify counts in a way that prevents probabilities from getting too low. The simplest algorithm here is Additive smoothing (aka [Lapace smoothing](https://en.wikipedia.org/wiki/Additive_smoothing)):

$$ P(w_t | prefix) = { Count(prefix, w_t) + \delta \over \sum_{\hat w} (Count(prefix, \hat w) + \delta) } $$

If counts for a given prefix are low, additive smoothing will adjust probabilities to a more uniform distribution. Not that the summation in the denominator goes over _all words in the vocabulary_.

Here's an example code we've implemented for you:

In [23]:
class LaplaceLanguageModel(NGramLanguageModel): 
    """ this code is an example, no need to change anything """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        counts = count_ngrams(lines, self.n)
        self.vocab = set(token for token_counts in counts.values() for token in token_counts)
        self.probs = defaultdict(Counter)

        for prefix in counts:
            token_counts = counts[prefix]
            total_count = sum(token_counts.values()) + delta * len(self.vocab)
            self.probs[prefix] = {token: (token_counts[token] + delta) / total_count
                                          for token in token_counts}
    def get_possible_next_tokens(self, prefix):
        token_probs = super().get_possible_next_tokens(prefix)
        missing_prob_total = 1.0 - sum(token_probs.values())
        missing_prob = missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        return {token: token_probs.get(token, missing_prob) for token in self.vocab}
    
    def get_next_token_prob(self, prefix, next_token):
        token_probs = super().get_possible_next_tokens(prefix)
        if next_token in token_probs:
            return token_probs[next_token]
        else:
            missing_prob_total = 1.0 - sum(token_probs.values())
            missing_prob_total = max(0, missing_prob_total) # prevent rounding errors
            return missing_prob_total / max(1, len(self.vocab) - len(token_probs))
        

**Disclaimer**: the implementation above assumes all words unknown within a given context to be equally likely, *as well as the words outside of vocabulary*. Therefore, its' perplexity will be lower than it should when encountering such words. Therefore, comparing it with a model with fewer unknown words will not be fair. When implementing your own smoothing, you may handle this by adding a virtual `UNK` token of non-zero probability. Technically, this will result in a model where probabilities do not add up to $1$, but it is close enough for a practice excercise.

In [24]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = LaplaceLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [25]:
for n in (1, 2, 3):
    lm = LaplaceLanguageModel(train_lines, n=n, delta=0.1)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))

C:\Users\user\AppData\Local\Temp\ipykernel_22028\1798100124.py:20: RuntimeWarning: divide by zero encountered in log
  log_proba = max(np.log(proba), min_logprob)


N = 1, Perplexity = 1832.66878
N = 2, Perplexity = 470.48021
N = 3, Perplexity = 3679.44765


In [ ]:
# optional: try to sample tokens from such a model

### Kneser-Ney smoothing (2 points)

Additive smoothing is simple, reasonably good but definitely not a State of The Art algorithm.


Your final task in this notebook is to implement [Kneser-Ney](https://en.wikipedia.org/wiki/Kneser%E2%80%93Ney_smoothing) smoothing.

It can be computed recurrently, for n>1:

$$P_{kn}(w_t | prefix_{n-1}) = { \max(0, Count(prefix_{n-1}, w_t) - \delta) \over \sum_{\hat w} Count(prefix_{n-1}, \hat w)} + \lambda_{prefix_{n-1}} \cdot P_{kn}(w_t | prefix_{n-2})$$

where
- $prefix_{n-1}$ is a tuple of {n-1} previous tokens
- $lambda_{prefix_{n-1}}$ is a normalization constant chosen so that probabilities add up to 1
- Unigram $P_{kn}(w_t | prefix_{n-2})$ corresponds to Kneser Ney smoothing for {N-1}-gram language model.
- Unigram $P_{kn}(w_t)$ is a special case: how likely it is to see x_t in an unfamiliar context

See lecture slides or wiki for more detailed formulae.

__Your task__ is to
- implement `KneserNeyLanguageModel` class,
- test it on 1-3 gram language models
- find optimal (within reason) smoothing delta for 3-gram language model with Kneser-Ney smoothing

In [ ]:
class KneserNeyLanguageModel(NGramLanguageModel): 
    """ A template for Kneser-Ney language model. Default delta may be suboptimal. """
    def __init__(self, lines, n, delta=1.0):
        self.n = n
        self.delta = delta
        
        self.counts = {num:defaultdict(Counter) for num in range(2,n+1)}
        
        for num in self.counts.keys():
            self.counts[num] = count_ngrams(lines, num)
            
    def pkn(self, prefix: str, num, next_token):
        prefix = tuple(prefix.split())
        p = max(0, self.counts[num][prefix][next_token] - self.delta) / sum(self.counts[num].values())
        
        lambda_ = (self.delta / self.counts[num][prefix]) * len(self.counts[num][prefix])
        
        
    def get_possible_next_tokens(self, prefix):
        pass
        
    def get_next_token_prob(self, prefix, next_token):
        pass

In [44]:
lm = KneserNeyLanguageModel(dummy_lines, 3)

print(lm.counts[3])

defaultdict(<class 'collections.Counter'>, {('_UNK_', '_UNK_'): Counter({'a': 13, 'the': 3, 'on': 3, 'using': 2, 'learning': 2, 'automatic': 2, 'why': 2, 'proceedings': 2, 'piecewise': 2, 'differential': 1, 'what': 1, 'p': 1, 'computational': 1, 'weak': 1, 'creating': 1, 'defeasible': 1, 'essence': 1, 'deep': 1, 'statistical': 1, 'complex': 1, 'serious': 1, 'preprocessing': 1, 'liquid': 1, 'mining': 1, 'towards': 1, 'icon': 1, 'recognition': 1, 'glottochronologic': 1, 'utility': 1, 'temporized': 1, 'backpropagation': 1, 'random': 1, 'network': 1, 'glottochronology': 1, 'time': 1, 'convolutional': 1, 'fitness': 1, 'flip': 1, 'autonomous': 1, 'activitynet': 1, 'decision': 1, 'text': 1, 'discrimination': 1, 'are': 1, 'extraction': 1, 'comments': 1, 'resource': 1, 'advances': 1, 'exploration': 1, 'quantified': 1, 'in': 1, 'introduction': 1, 'beyond': 1, 'norm': 1, 'about': 1, 'unary': 1, 'some': 1, 'convex': 1, 'neurocontrol': 1, 'philosophy': 1, 'parallels': 1, 'an': 1, 'calculate': 1, 'g

In [ ]:
#test that it's a valid probability model
for n in (1, 2, 3):
    dummy_lm = KneserNeyLanguageModel(dummy_lines, n=n)
    assert np.allclose(sum([dummy_lm.get_next_token_prob('a', w_i) for w_i in dummy_lm.vocab]), 1), "I told you not to break anything! :)"

In [ ]:
for n in (1, 2, 3):
    lm = KneserNeyLanguageModel(train_lines, n=n, smoothing=<...>)
    ppx = perplexity(lm, test_lines)
    print("N = %i, Perplexity = %.5f" % (n, ppx))